# Morphology experiments

In [1]:
# set the path of this jupyter notebook
import sys
import os
cwd = os.getcwd()
parent_dir = os.path.dirname(cwd)
parent_dir = os.path.dirname(parent_dir)
cwd = parent_dir
sys.path.append(parent_dir)
project_root = parent_dir

# import packages
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import os
from src.dataGenerator.spherical_data_generator import *
from src.utils.sht_helper import *
from src.stochastics.sde import *
from src.stochastics.sde_solver import *
import matplotlib.pyplot as plt
import random
import src.training.trainer as Trainer
from flax.training import checkpoints
from src.models.neural_operator import SCUNO
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
def get_random_int():
    return random.randint(0, 1000000)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
jax.clear_caches()


train_steps = 1000
in_grid_L = 12 # Training data resolution




In [4]:
# 2D manifold data generator

s_aqua_data_generator_X0 = S2ManifoldDataGenerator(manifold_type="real_data", seed=get_random_int(), randomization=False, file_path=project_root + '/data/morph/Sylvilagus_aquaticus.stl', scale=0.5, src_type='mesh')
x0 = s_aqua_data_generator_X0.generate_data(in_grid_L, 5)

s_obs_data_generator_XT = S2ManifoldDataGenerator(manifold_type="real_data", seed=get_random_int(), randomization=False, file_path=project_root + '/data/morph/Sylvilagus_obscurus.stl', scale=0.5, src_type='mesh')
xT = s_obs_data_generator_XT.generate_data(in_grid_L, 1)


In [5]:
sde_3d = Kunita_Flow_SDE_3D_Eulerian_2Dmanifold_distance(k_alpha= 1.0, k_sigma=0.4, grid_num=10, grid_range=[-1,1], x0=x0[0])
sde_solver = EulerMaruyama.from_sde(sde_3d, 0.02, 1.0, 3, None,debug_mode=False)
xs,_ = sde_solver.solve(x0[0], rng_key=jrandom.PRNGKey(get_random_int()))

(1000, 3)
(12, 23, 3)


In [ ]:
model = SCUNO(x_feature_dim=3, l_list=(8,8), lift_dim=16, latent_feature_dims=(2,2), sampling="gl", activation="gelu")
trainer = Trainer.NeuralOpTrainer(seed=get_random_int(), landmark_num=in_grid_L)

# set the checkpoint path and the retrain checkpoint path
checkpoint_path = project_root + '/checkpoints/lepus_model'

# detect if the checkpoint exists, if not, train the model
if not os.path.exists(checkpoint_path):
    train_state = trainer.train_state_init(model, lr=5e-4, model_kwargs={'x': jax.random.normal(jrandom.PRNGKey(get_random_int()), x0[0].shape), 't': jnp.array([0]),'object_fn': 'Yang', 'x_L': in_grid_L})
    train_state, train_loss = trainer.train(train_state, sde_3d, sde_solver, s_aqua_data_generator_X0, train_steps, 16, in_grid_L)
    plt.plot(train_loss)
    plt.show()
    # save the model
    config = {"dimension": x0[0].shape}
    ckpt = {"model": train_state, "config": config}
    checkpoints.save_checkpoint(checkpoint_path, ckpt, step=train_steps, overwrite=True, keep=1)
else:
    restored_checkpoint = checkpoints.restore_checkpoint(checkpoint_path, target=None)
    params = restored_checkpoint["model"]["params"]
    train_state = trainer.train_state_init(model, lr=1e-4, model_kwargs={'x': jax.random.normal(jrandom.PRNGKey(get_random_int()), x0[0].shape), 't': jnp.array([0]),'object_fn': 'Yang', 'x_L': in_grid_L}, retrain=True, ckpt_params=params)
    print("found restored checkpoint, start inferencing...")



Training neural operator:   0%|          | 0/1000 [00:00<?, ?it/s]

(1000, 3)
(12, 23, 3)
(1000, 3)
(12, 23, 3)
x.shape (12, 23, 3)
x_prev.shape (12, 23, 3)
drift_prev.shape (12, 23, 3)
dt.shape ()


Training loss: 1.6716487945564418:  40%|███▉      | 399/1000 [05:35<08:14,  1.21it/s]

In [ ]:
test_L = 40
score_fn = lambda x, t, x0: model.apply(train_state.params, x, t, test_L) # x0 is not used





